In [ ]:
import requests
import pandas as pd
from pathlib import Path

In [ ]:
BASE_URL = "https://opendataapi.dmi.dk/v2/climateData/collections/stationValue/items"
parameters = ["mean_radiation",
    "mean_wind_speed",
    "max_wind_speed_3sec",
    "max_wind_speed_10min"
]
# List to collect DataFrames
dfs = []
for col in parameters:
    # Parameters
    params = {
        "stationId": "06170",                     # your station (ROSKILDE LUFTHAVN)
        "parameterId": col,                       # parameter to retrieve
        "timeResolution": "hour",                 # hourly values
        "datetime": "2024-12-01T00:00:00Z/2025-09-30T23:59:59Z",
        "limit": 30000 ,                           # max records
    }
    # Send request
    response = requests.get(BASE_URL, params=params)
    # Check success
    if response.status_code == 200:
        data = response.json()
        # Each record is a GeoJSON feature
        features = data.get("features", [])
        # Parse all stationValue fields into a list of dicts
        records = []
        for f in features:
            props = f.get("properties", {})
            geom = f.get("geometry", {})
            coords = geom.get("coordinates", None)
            # Include geometry if you want
            record = {
                **props,
                "longitude": coords[0] if coords else None,
                "latitude": coords[1] if coords else None
            }
            records.append(record)
        # Convert to DataFrame and add to list if not empty
        if records:
            df_param = pd.DataFrame(records)
            dfs.append(df_param)
    else:
        print("Error", response.status_code, response.text)
# Stack all DataFrames on top of each other
if dfs:
    df_stacked = pd.concat(dfs, ignore_index=True)
else:
    print("No valid data returned for any parameter")

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
df_stacked.to_csv(DATA_DIR / "DMI_API.csv")